# Information Retrieval and NLP: Comparing Seven Retrieval Methods

This notebook ranks the same small corpus against the same query with seven methods, from exact keyword matching to transformer embeddings, so their behaviour can be compared side by side.

| | Method | Idea |
|---|---|---|
| A | Boolean retrieval | Exact match with AND / OR / NOT on an inverted index |
| B | TF-IDF | Weighted keyword overlap, ranked by cosine similarity |
| C | Word2Vec (CBOW, Skip-gram) | Word vectors learned from local context |
| D | GloVe | Pretrained vectors from global co-occurrence |
| E | FastText | Word vectors built from character n-grams |
| F | Sentence-BERT | Sentence embeddings from `all-MiniLM-L6-v2` |
| G | Transformer | Mean-pooled `distilbert-base-uncased` embeddings |

**Corpus:** 8 short documents on AI and search. **Query:** *"neural networks for language understanding"*.

> The corpus is deliberately tiny, so the scores illustrate how each method behaves; they are not a benchmark.

## Setup
Install the libraries once (CPU is enough). GloVe, Sentence-BERT and DistilBERT are downloaded on the first run and cached.

In [ ]:
# !pip install scikit-learn gensim sentence-transformers transformers torch --quiet

Each optional library is imported inside `try`, so a missing package skips only its own section.

In [1]:
import re
import string
import math
from collections import defaultdict

import numpy as np

# --- scikit-learn ---
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print("[WARN] scikit-learn not found. TF-IDF section will be skipped.")

# --- gensim (Word2Vec, FastText) ---
try:
    from gensim.models import Word2Vec, FastText
    GENSIM_OK = True
except ImportError:
    GENSIM_OK = False
    print("[WARN] gensim not found. Word2Vec / FastText sections will be skipped.")

# --- sentence-transformers (BERT) ---
try:
    from sentence_transformers import SentenceTransformer, util as st_util
    SBERT_OK = True
except ImportError:
    SBERT_OK = False
    print("[WARN] sentence-transformers not found. BERT section will be skipped.")

# --- HuggingFace transformers (raw Transformer embeddings) ---
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    HF_OK = True
except ImportError:
    HF_OK = False
    print("[WARN] transformers / torch not found. Transformer section will be skipped.")

## 1. Corpus and query

In [2]:
DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence focused on learning from data.",
    "Deep learning uses neural networks with many layers to model complex patterns.",
    "Natural language processing helps computers understand and generate human language.",
    "Information retrieval systems search and rank documents based on a user query.",
    "Python is a popular programming language widely used in data science and AI.",
    "Transformers revolutionized NLP by using self-attention mechanisms for language models.",
    "Word embeddings represent words as dense vectors capturing semantic relationships.",
    "Search engines use indexing and ranking algorithms to retrieve relevant web pages.",
]

QUERY = "neural networks for language understanding"

print("=" * 70)
print("  NLP & INFORMATION RETRIEVAL – CLASSROOM PROJECT")
print("=" * 70)
print(f"\nCorpus: {len(DOCUMENTS)} documents")
print(f"Query : \"{QUERY}\"\n")
for i, doc in enumerate(DOCUMENTS):
    print(f"  D{i}: {doc}")

  NLP & INFORMATION RETRIEVAL – CLASSROOM PROJECT

Corpus: 8 documents
Query : "neural networks for language understanding"

  D0: Machine learning is a subset of artificial intelligence focused on learning from data.
  D1: Deep learning uses neural networks with many layers to model complex patterns.
  D2: Natural language processing helps computers understand and generate human language.
  D3: Information retrieval systems search and rank documents based on a user query.
  D4: Python is a popular programming language widely used in data science and AI.
  D5: Transformers revolutionized NLP by using self-attention mechanisms for language models.
  D6: Word embeddings represent words as dense vectors capturing semantic relationships.
  D7: Search engines use indexing and ranking algorithms to retrieve relevant web pages.


## 2. Preprocessing
Lowercase, remove punctuation, split on whitespace and drop stop words. Boolean retrieval, Word2Vec, GloVe and FastText use these tokens; TF-IDF and the transformer models use their own tokenizers.

In [3]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "in", "on", "at", "to", "for", "of", "and", "or", "not", "with",
    "as", "by", "from", "that", "this", "it", "its", "has", "have",
    "had", "do", "does", "did", "will", "would", "can", "could",
    "which", "who", "how", "their", "they", "them", "many", "using",
}

def preprocess(text: str) -> list[str]:
    """Lowercase → strip punctuation → tokenize → remove stopwords."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS]

# Tokenised corpus (list-of-lists)
tokenised_corpus = [preprocess(doc) for doc in DOCUMENTS]

## A. Boolean retrieval
An inverted index maps each term to the set of documents that contain it. Queries are answered with set operations, so a document either matches or it does not: there is no ranking.

In [4]:
print("\n" + "=" * 70)
print("  A.  BOOLEAN RETRIEVAL")
print("=" * 70)

# ── Build an inverted index  {term → set of doc IDs} ────────────────────────
inverted_index: dict[str, set] = defaultdict(set)
for doc_id, tokens in enumerate(tokenised_corpus):
    for token in tokens:
        inverted_index[token].add(doc_id)

ALL_DOCS = set(range(len(DOCUMENTS)))

def get_postings(term: str) -> set:
    """Return the posting list for a single term (empty set if missing)."""
    return inverted_index.get(term.lower(), set())

def boolean_AND(*terms) -> set:
    result = ALL_DOCS.copy()
    for t in terms:
        result &= get_postings(t)
    return result

def boolean_OR(*terms) -> set:
    result = set()
    for t in terms:
        result |= get_postings(t)
    return result

def boolean_NOT(term: str, universe: set = ALL_DOCS) -> set:
    return universe - get_postings(term)

def boolean_AND_NOT(term_a: str, term_b: str) -> set:
    return get_postings(term_a) - get_postings(term_b)

def show_bool_result(description: str, result: set):
    ids = sorted(result)
    print(f"\n  Query : {description}")
    if ids:
        for i in ids:
            print(f"    → D{i}: {DOCUMENTS[i]}")
    else:
        print("    → No matching documents.")

# ── Example Boolean queries ───────────────────────────────────────────────────
show_bool_result(
    "language AND learning",
    boolean_AND("language", "learning")
)
show_bool_result(
    "neural OR transformers",
    boolean_OR("neural", "transformers")
)
show_bool_result(
    "NOT python",
    boolean_NOT("python")
)
show_bool_result(
    "language AND NOT neural",
    boolean_AND_NOT("language", "neural")
)
show_bool_result(
    "data OR search",
    boolean_OR("data", "search")
)

  A.  BOOLEAN RETRIEVAL

  Query : language AND learning
    → No matching documents.

  Query : neural OR transformers
    → D1: Deep learning uses neural networks with many layers to model complex patterns.
    → D5: Transformers revolutionized NLP by using self-attention mechanisms for language models.

  Query : NOT python
    → D0: Machine learning is a subset of artificial intelligence focused on learning from data.
    → D1: Deep learning uses neural networks with many layers to model complex patterns.
    → D2: Natural language processing helps computers understand and generate human language.
    → D3: Information retrieval systems search and rank documents based on a user query.
    → D5: Transformers revolutionized NLP by using self-attention mechanisms for language models.
    → D6: Word embeddings represent words as dense vectors capturing semantic relationships.
    → D7: Search engines use indexing and ranking algorithms to retrieve relevant web pages.

  Query : languag

## B. TF-IDF vector space model
Each document becomes a vector of term weights (frequent in the document, rare in the corpus). Documents are ranked by cosine similarity to the query vector.

In [5]:
print("\n" + "=" * 70)
print("  B.  TF-IDF VECTOR SPACE MODEL")
print("=" * 70)

if SKLEARN_OK:
    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(DOCUMENTS)       # shape: (8, vocab_size)

    query_vec = tfidf.transform([QUERY])                # shape: (1, vocab_size)
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    print(f"\n  Query: \"{QUERY}\"")
    print(f"  {'Rank':<6} {'Doc':<5} {'Score':>8}   Document")
    print(f"  {'-'*4}   {'-'*3}   {'-'*8}   {'-'*45}")
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"  {rank:<6} D{doc_id:<4} {score:>8.4f}   {DOCUMENTS[doc_id][:60]}...")
else:
    print("  [SKIPPED] scikit-learn is not installed.")

  B.  TF-IDF VECTOR SPACE MODEL

  Query: "neural networks for language understanding"
  Rank   Doc      Score   Document
  ----   ---   --------   ---------------------------------------------
  1      D1      0.4268   Deep learning uses neural networks with many layers to model...
  2      D2      0.2184   Natural language processing helps computers understand and g...
  3      D4      0.1148   Python is a popular programming language widely used in data...
  4      D5      0.1128   Transformers revolutionized NLP by using self-attention mech...
  5      D0      0.0000   Machine learning is a subset of artificial intelligence focu...
  6      D3      0.0000   Information retrieval systems search and rank documents base...
  7      D6      0.0000   Word embeddings represent words as dense vectors capturing s...
  8      D7      0.0000   Search engines use indexing and ranking algorithms to retrie...


## C. Word2Vec: CBOW and Skip-gram
Word vectors trained on the corpus itself. CBOW predicts a word from its context; Skip-gram predicts the context from the word. The corpus is repeated 50 times so the tiny model has enough training examples, so its neighbours mostly reflect which words share a sentence.

In [6]:
print("\n" + "=" * 70)
print("  C.  WORD2VEC  (CBOW  &  SKIP-GRAM)")
print("=" * 70)

if GENSIM_OK:
    # gensim requires a list-of-lists corpus of tokens
    # We repeat the corpus several times so there is enough context to train
    training_corpus = tokenised_corpus * 50

    SAMPLE_WORDS = ["language", "learning", "neural", "data"]

    for model_name, sg_flag in [("CBOW (sg=0)", 0), ("Skip-gram (sg=1)", 1)]:
        print(f"\n  ── {model_name} ──")
        model_w2v = Word2Vec(
            sentences=training_corpus,
            vector_size=50,   # small dimension — fast to train
            window=3,         # context window size
            min_count=1,      # keep all words (tiny corpus)
            sg=sg_flag,       # 0 = CBOW, 1 = skip-gram
            epochs=50,
            seed=42,
        )

        for word in SAMPLE_WORDS:
            if word in model_w2v.wv:
                neighbors = model_w2v.wv.most_similar(word, topn=3)
                pairs = ", ".join(f"{w}({s:.3f})" for w, s in neighbors)
                print(f"    nearest(\"{word}\") → {pairs}")
            else:
                print(f"    \"{word}\" not in vocabulary")
else:
    print("  [SKIPPED] gensim is not installed.")

  C.  WORD2VEC  (CBOW  &  SKIP-GRAM)

  ── CBOW (sg=0) ──
    nearest("language") → human(0.835), processing(0.831), natural(0.831)
    nearest("learning") → machine(0.919), intelligence(0.917), focused(0.916)
    nearest("neural") → deep(0.997), networks(0.996), uses(0.995)
    nearest("data") → ai(0.881), science(0.873), used(0.870)

  ── Skip-gram (sg=1) ──
    nearest("language") → human(0.751), processing(0.743), helps(0.742)
    nearest("learning") → intelligence(0.890), subset(0.890), machine(0.889)
    nearest("neural") → model(0.994), deep(0.993), uses(0.993)
    nearest("data") → focused(0.863), ai(0.856), artificial(0.847)


## D. GloVe
Pretrained 50-dimensional vectors (`glove-wiki-gigaword-50`). A document vector is the mean of its word vectors, ranked by cosine similarity to the query. If the download fails, a small hand-made example runs instead.

In [7]:
print("\n" + "=" * 70)
print("  D.  GloVe EMBEDDINGS")
print("=" * 70)

# gensim-downloader can pull small GloVe vectors, but the smallest bundle
# ('glove-twitter-25') is ~100 MB and requires internet access.
# We attempt the download; if it fails we show a clean fallback.

GLOVE_LOADED = False

if GENSIM_OK:
    try:
        import gensim.downloader as gensim_api
        print("  Attempting to load glove-wiki-gigaword-50 (≈66 MB, cached after first run)…")
        glove = gensim_api.load("glove-wiki-gigaword-50")
        GLOVE_LOADED = True
        print("  GloVe model loaded successfully.")
    except Exception as e:
        print(f"  [INFO] Could not download GloVe: {e}")
        print("  Falling back to a manually simulated mini-GloVe demo.")
else:
    print("  [SKIPPED] gensim is not installed.")

if GLOVE_LOADED:
    print(f"\n  Query: \"{QUERY}\"")
    # Rank corpus documents using mean-pooled GloVe vectors
    def sentence_vec_glove(tokens, model):
        vecs = [model[t] for t in tokens if t in model]
        return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

    query_tokens = preprocess(QUERY)
    q_vec = sentence_vec_glove(query_tokens, glove)

    scores = []
    for i, tokens in enumerate(tokenised_corpus):
        d_vec = sentence_vec_glove(tokens, glove)
        if q_vec.any() and d_vec.any():
            sim = np.dot(q_vec, d_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(d_vec) + 1e-10)
        else:
            sim = 0.0
        scores.append((i, sim))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    print(f"  {'Rank':<6} {'Doc':<5} {'Score':>8}   Document")
    print(f"  {'-'*4}   {'-'*3}   {'-'*8}   {'-'*45}")
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"  {rank:<6} D{doc_id:<4} {score:>8.4f}   {DOCUMENTS[doc_id][:60]}...")

    # Also show nearest neighbors for a word
    print("\n  Nearest GloVe neighbors for 'language':")
    for w, s in glove.most_similar("language", topn=5):
        print(f"    {w:<20} {s:.4f}")

elif GENSIM_OK and not GLOVE_LOADED:
    # ── Minimal manual GloVe-style demo (no download needed) ─────────────────
    print("\n  [FALLBACK DEMO] Using tiny hand-crafted GloVe-style vectors")
    print("  In production, use: gensim.downloader.load('glove-wiki-gigaword-50')")
    print("  or download GloVe text files from https://nlp.stanford.edu/projects/glove/")

    mini_glove = {
        "language":  np.array([0.9, 0.2, 0.1, 0.5, 0.3]),
        "learning":  np.array([0.8, 0.5, 0.3, 0.4, 0.2]),
        "neural":    np.array([0.7, 0.8, 0.9, 0.1, 0.2]),
        "network":   np.array([0.6, 0.7, 0.8, 0.2, 0.3]),
        "data":      np.array([0.5, 0.4, 0.2, 0.8, 0.7]),
        "machine":   np.array([0.7, 0.6, 0.3, 0.5, 0.4]),
        "deep":      np.array([0.6, 0.9, 0.8, 0.1, 0.1]),
        "model":     np.array([0.8, 0.7, 0.5, 0.4, 0.3]),
        "text":      np.array([0.9, 0.3, 0.1, 0.6, 0.4]),
        "search":    np.array([0.3, 0.2, 0.1, 0.9, 0.8]),
        "retrieval": np.array([0.2, 0.3, 0.2, 0.8, 0.9]),
    }

    def mini_glove_sim(w1, w2):
        if w1 in mini_glove and w2 in mini_glove:
            v1, v2 = mini_glove[w1], mini_glove[w2]
            return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
        return None

    print("\n  Word pair cosine similarities (mini GloVe):")
    pairs = [("language", "learning"), ("neural", "network"),
             ("data", "retrieval"), ("neural", "search")]
    for w1, w2 in pairs:
        s = mini_glove_sim(w1, w2)
        print(f"    sim({w1:10s}, {w2:10s}) = {s:.4f}" if s is not None else f"    OOV: {w1} / {w2}")

  D.  GloVe EMBEDDINGS
  Attempting to load glove-wiki-gigaword-50 (≈66 MB, cached after first run)…
  GloVe model loaded successfully.

  Query: "neural networks for language understanding"
  Rank   Doc      Score   Document
  ----   ---   --------   ---------------------------------------------
  1      D1      0.8842   Deep learning uses neural networks with many layers to model...
  2      D0      0.8665   Machine learning is a subset of artificial intelligence focu...
  3      D2      0.8523   Natural language processing helps computers understand and g...
  4      D4      0.8401   Python is a popular programming language widely used in data...
  5      D5      0.7869   Transformers revolutionized NLP by using self-attention mech...
  6      D3      0.7631   Information retrieval systems search and rank documents base...
  7      D6      0.7543   Word embeddings represent words as dense vectors capturing s...
  8      D7      0.7140   Search engines use indexing and ranking algori

## E. FastText
Like Word2Vec, but each word is also represented by its character n-grams, so the model can build vectors for misspelled or unseen words.

In [8]:
print("\n" + "=" * 70)
print("  E.  FASTTEXT  (subword embeddings)")
print("=" * 70)

if GENSIM_OK:
    ft_model = FastText(
        sentences=training_corpus,
        vector_size=50,
        window=3,
        min_count=1,
        epochs=50,
        seed=42,
    )

    print("  FastText model trained on toy corpus.\n")

    # Regular words
    for word in ["language", "learning"]:
        nbrs = ft_model.wv.most_similar(word, topn=3)
        pairs = ", ".join(f"{w}({s:.3f})" for w, s in nbrs)
        print(f"  nearest(\"{word}\") → {pairs}")

    # OOV / rare-word demo — FastText handles these via subword n-grams
    oov_words = ["languaging", "learnt", "neurall"]
    print("\n  Out-of-vocabulary / rare word handling (FastText subword magic):")
    for oov in oov_words:
        vec = ft_model.wv[oov]      # Does NOT crash – uses character n-grams
        nbrs = ft_model.wv.most_similar(oov, topn=3)
        pairs = ", ".join(f"{w}({s:.3f})" for w, s in nbrs)
        print(f"    \"{oov}\" (OOV) → nearest: {pairs}")
        print(f"      vector norm = {np.linalg.norm(vec):.4f}  (non-zero → subword worked)")
else:
    print("  [SKIPPED] gensim is not installed.")

  E.  FASTTEXT  (subword embeddings)
  FastText model trained on toy corpus.

  nearest("language") → processing(0.867), understand(0.855), human(0.855)
  nearest("learning") → machine(0.931), artificial(0.929), intelligence(0.927)

  Out-of-vocabulary / rare word handling (FastText subword magic):
    "languaging" (OOV) → nearest: language(0.934), processing(0.899), understand(0.880)
      vector norm = 2.3408  (non-zero → subword worked)
    "learnt" (OOV) → nearest: learning(0.984), artificial(0.915), machine(0.914)
      vector norm = 1.8738  (non-zero → subword worked)
    "neurall" (OOV) → nearest: neural(0.996), complex(0.993), layers(0.992)
      vector norm = 2.3185  (non-zero → subword worked)


## F. Sentence-BERT
`all-MiniLM-L6-v2` is a transformer trained specifically to produce sentence embeddings for semantic similarity.

In [9]:
print("\n" + "=" * 70)
print("  F.  BERT  (sentence-transformers)")
print("=" * 70)

if SBERT_OK:
    print("  Loading 'all-MiniLM-L6-v2' (≈90 MB, cached after first run)…")
    sbert = SentenceTransformer("all-MiniLM-L6-v2")

    doc_embeddings = sbert.encode(DOCUMENTS, convert_to_tensor=True)
    query_embedding = sbert.encode(QUERY,     convert_to_tensor=True)

    cos_scores = st_util.cos_sim(query_embedding, doc_embeddings)[0]
    ranked = sorted(enumerate(cos_scores.tolist()), key=lambda x: x[1], reverse=True)

    print(f"\n  Query: \"{QUERY}\"")
    print(f"  {'Rank':<6} {'Doc':<5} {'Score':>8}   Document")
    print(f"  {'-'*4}   {'-'*3}   {'-'*8}   {'-'*45}")
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"  {rank:<6} D{doc_id:<4} {score:>8.4f}   {DOCUMENTS[doc_id][:60]}...")
else:
    print("  [SKIPPED] sentence-transformers is not installed.")

  F.  BERT  (sentence-transformers)
  Loading 'all-MiniLM-L6-v2' (≈90 MB, cached after first run)…

  Query: "neural networks for language understanding"
  Rank   Doc      Score   Document
  ----   ---   --------   ---------------------------------------------
  1      D6      0.4400   Word embeddings represent words as dense vectors capturing s...
  2      D2      0.4296   Natural language processing helps computers understand and g...
  3      D5      0.4201   Transformers revolutionized NLP by using self-attention mech...
  4      D1      0.4096   Deep learning uses neural networks with many layers to model...
  5      D0      0.1795   Machine learning is a subset of artificial intelligence focu...
  6      D3      0.1226   Information retrieval systems search and rank documents base...
  7      D7      0.0823   Search engines use indexing and ranking algorithms to retrie...
  8      D4      0.0403   Python is a popular programming language widely used in data...


## G. Transformer with mean pooling
`distilbert-base-uncased` is a general language model, not trained for similarity. Its token embeddings are averaged (padding masked out) to get one vector per text.

In [10]:
print("\n" + "=" * 70)
print("  G.  TRANSFORMER  (HuggingFace – mean pooling)")
print("=" * 70)

if HF_OK:
    HF_MODEL = "distilbert-base-uncased"   # smaller, faster BERT variant (~66M parameters)
    print(f"  Loading '{HF_MODEL}' (≈268 MB, cached after first run)…")

    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
    hf_model  = AutoModel.from_pretrained(HF_MODEL)
    hf_model.eval()

    def mean_pool(text: str) -> np.ndarray:
        """Tokenise → forward pass → average over all token embeddings (incl. [CLS]/[SEP])."""
        inputs = tokenizer(text, return_tensors="pt",
                           truncation=True, max_length=128, padding=True)
        with torch.no_grad():
            outputs = hf_model(**inputs)
        # outputs.last_hidden_state: (1, seq_len, hidden_size)
        token_embeddings = outputs.last_hidden_state.squeeze(0)    # (seq_len, H)
        attention_mask   = inputs["attention_mask"].squeeze(0)     # (seq_len,)
        # Mask padding tokens before averaging
        mask_expanded = attention_mask.unsqueeze(-1).float()
        pooled = (token_embeddings * mask_expanded).sum(0) / mask_expanded.sum(0)
        return pooled.numpy()

    q_emb    = mean_pool(QUERY)
    d_embs   = np.stack([mean_pool(doc) for doc in DOCUMENTS])

    # Cosine similarity (manual, no sklearn needed here)
    def cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    scores   = [(i, cosine(q_emb, d_embs[i])) for i in range(len(DOCUMENTS))]
    ranked   = sorted(scores, key=lambda x: x[1], reverse=True)

    print(f"\n  Query: \"{QUERY}\"")
    print(f"  Model : {HF_MODEL}  (mean-pooled token embeddings)")
    print(f"  {'Rank':<6} {'Doc':<5} {'Score':>8}   Document")
    print(f"  {'-'*4}   {'-'*3}   {'-'*8}   {'-'*45}")
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"  {rank:<6} D{doc_id:<4} {score:>8.4f}   {DOCUMENTS[doc_id][:60]}...")
else:
    print("  [SKIPPED] transformers / torch is not installed.")

  G.  TRANSFORMER  (HuggingFace – mean pooling)
  Loading 'distilbert-base-uncased' (≈268 MB, cached after first run)…

  Query: "neural networks for language understanding"
  Model : distilbert-base-uncased  (mean-pooled token embeddings)
  Rank   Doc      Score   Document
  ----   ---   --------   ---------------------------------------------
  1      D2      0.8614   Natural language processing helps computers understand and g...
  2      D1      0.8590   Deep learning uses neural networks with many layers to model...
  3      D5      0.8348   Transformers revolutionized NLP by using self-attention mech...
  4      D0      0.8035   Machine learning is a subset of artificial intelligence focu...
  5      D6      0.7625   Word embeddings represent words as dense vectors capturing s...
  6      D3      0.7381   Information retrieval systems search and rank documents base...
  7      D4      0.7234   Python is a popular programming language widely used in data...
  8      D7      0.6872

## Results on this query

Top three documents for *"neural networks for language understanding"* (from the outputs above):

| Method | 1st | 2nd | 3rd | Score range (all 8 docs) |
|---|---|---|---|---|
| TF-IDF | D1 (0.43) | D2 (0.22) | D4 (0.11) | 0.00 – 0.43 |
| GloVe (mean of word vectors) | D1 (0.88) | D0 (0.87) | D2 (0.85) | 0.71 – 0.88 |
| Sentence-BERT | D6 (0.44) | D2 (0.43) | D5 (0.42) | 0.04 – 0.44 |
| DistilBERT (mean pooling) | D2 (0.86) | D1 (0.86) | D5 (0.83) | 0.69 – 0.86 |

D0 = machine learning · D1 = deep learning / neural networks · D2 = natural language processing · D4 = Python · D5 = transformers / NLP · D6 = word embeddings

**Observations**

- **Boolean retrieval** returns nothing for `language AND learning`: no single document has both words, which shows how strict exact matching is.
- **TF-IDF** gives a score of 0 to four of the eight documents, because they share no word with the query. It ranks D4 (Python) third only because the word *language* appears in it.
- **GloVe** and **DistilBERT** give every document a high score (above 0.68), so the gaps between relevant and irrelevant documents are small. Averaged vectors and general-purpose models tend to be similar to everything.
- **Sentence-BERT** separates the documents best: the four AI and language documents score about 0.41–0.44, and the search and Python documents fall to 0.04–0.12. This is what a model trained for semantic similarity adds.
- **FastText** still finds sensible neighbours for unseen words such as *learnt* → *learning* and *neurall* → *neural*.
- Word2Vec and FastText neighbours come from 8 sentences, so they mostly reflect words that appear in the same sentence, not general meaning.

## Summary of the methods

In [11]:
print("\n" + "=" * 70)
print("  SUMMARY – Method Comparison")
print("=" * 70)

SUMMARY = [
    ("Boolean Retrieval",
     "Exact keyword matching using AND / OR / NOT logic.\n"
     "     No ranking — a document either matches or it does not.\n"
     "     Fast and interpretable; breaks for synonyms or paraphrases."),

    ("TF-IDF (Vector Space Model)",
     "Weighted lexical matching using term frequency and inverse\n"
     "     document frequency. Documents are ranked by cosine similarity\n"
     "     to the query vector. Handles term importance but ignores meaning."),

    ("Word2Vec / CBOW",
     "Learns dense word vectors by predicting a target word from its\n"
     "     surrounding context window. Captures distributional semantics\n"
     "     (words with similar contexts get similar vectors)."),

    ("Word2Vec / Skip-gram",
     "Reverse of CBOW: predicts context words given a target word.\n"
     "     Better for rare words and larger datasets; slower to train."),

    ("GloVe",
     "Trains on global word co-occurrence statistics over the whole\n"
     "     corpus. Combines benefits of matrix factorisation and Word2Vec.\n"
     "     Pretrained vectors encode broad semantic relationships."),

    ("FastText",
     "Extends Word2Vec with character-level subword n-grams.\n"
     "     Can produce meaningful vectors for OOV / misspelled words.\n"
     "     Great for morphologically rich languages and noisy text."),

    ("BERT (sentence-transformers)",
     "Transformer model fine-tuned to produce fixed-size sentence\n"
     "     embeddings. Context-aware: the same word gets different vectors\n"
     "     depending on surrounding words. Strong for semantic similarity."),

    ("Transformer (HuggingFace mean pooling)",
     "Raw pretrained transformer (DistilBERT) with mean pooling\n"
     "     over token embeddings. Demonstrates how general-purpose LM\n"
     "     representations can be used for retrieval without fine-tuning."),
]

for method, description in SUMMARY:
    print(f"\n  ✦  {method}")
    print(f"     {description}")

print("\n" + "=" * 70)
print("  END OF PROJECT")
print("=" * 70)

  SUMMARY – Method Comparison

  ✦  Boolean Retrieval
     Exact keyword matching using AND / OR / NOT logic.
     No ranking — a document either matches or it does not.
     Fast and interpretable; breaks for synonyms or paraphrases.

  ✦  TF-IDF (Vector Space Model)
     Weighted lexical matching using term frequency and inverse
     document frequency. Documents are ranked by cosine similarity
     to the query vector. Handles term importance but ignores meaning.

  ✦  Word2Vec / CBOW
     Learns dense word vectors by predicting a target word from its
     surrounding context window. Captures distributional semantics
     (words with similar contexts get similar vectors).

  ✦  Word2Vec / Skip-gram
     Reverse of CBOW: predicts context words given a target word.
     Better for rare words and larger datasets; slower to train.

  ✦  GloVe
     Trains on global word co-occurrence statistics over the whole
     corpus. Combines benefits of matrix factorisation and Word2Vec.
     Pretr